# Exploring the White House TikTok Classifications

600 posts, Aug 19 2025 – Apr 1 2026. Each classified by subject (8 categories), packaging (7 levels), and technique tags (60+). 

See CLASSIFICATION_MATRIX.md for details on the tagging logic.

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, HTML

WAR_MEMES = Path("/Users/wongpeiting/Desktop/CU/python-work/war-memes")
PEAK_MEME = Path("/Users/wongpeiting/Desktop/CU/python-work/peak-meme")

# Load grid_posts.json (has subject, packaging, packaging_level, tags, views)
with open(PEAK_MEME / "data" / "grid_posts.json") as f:
    posts = json.load(f)

df = pd.DataFrame(posts)
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M")
df["post_num"] = range(1, len(df) + 1)
df["is_meme"] = df["packaging_level"] >= 5
df["is_war"] = df["subject"] == "war"
df["is_war_meme"] = df["is_war"] & df["is_meme"]

# Explode tags into a flat list for tag analysis
tag_rows = df.explode("tags").rename(columns={"tags": "tag"})

print(f"{len(df)} posts | {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Total views: {df['views'].sum():,.0f} ({df['views'].sum()/1e9:.2f}B)")
print(f"Unique subjects: {df['subject'].nunique()} | Packaging levels: {df['packaging_level'].nunique()}")
print(f"Unique tags: {tag_rows['tag'].nunique()}")

600 posts | 2025-08-19 to 2026-04-01
Total views: 1,453,171,000 (1.45B)
Unique subjects: 8 | Packaging levels: 7
Unique tags: 30


## Subject breakdown

In [2]:
subj = df.groupby("subject").agg(
    count=("id", "size"),
    total_views=("views", "sum"),
    avg_views=("views", "mean"),
    median_views=("views", "median"),
).sort_values("count", ascending=False)

subj["pct_posts"] = (subj["count"] / len(df) * 100).round(1)
subj["pct_views"] = (subj["total_views"] / df["views"].sum() * 100).round(1)
subj["avg_views"] = subj["avg_views"].apply(lambda x: f"{x/1e6:.1f}M")
subj["median_views"] = subj["median_views"].apply(lambda x: f"{x/1e6:.1f}M")
subj["total_views"] = subj["total_views"].apply(lambda x: f"{x/1e6:.0f}M")

print("SUBJECT BREAKDOWN")
print("="*70)
display(subj[["count", "pct_posts", "total_views", "pct_views", "avg_views", "median_views"]])

SUBJECT BREAKDOWN


,count,pct_posts,total_views,pct_views,avg_views,median_views
subject,,,,,,
governance,149,24.8,232M,16.0,1.6M,0.8M
opponents,83,13.8,131M,9.0,1.6M,0.9M
promotion,83,13.8,149M,10.3,1.8M,1.1M
trump,79,13.2,183M,12.6,2.3M,1.6M
enforcement,70,11.7,265M,18.3,3.8M,1.1M
culture,68,11.3,155M,10.7,2.3M,1.2M
war,64,10.7,335M,23.0,5.2M,2.2M
unclear,4,0.7,3M,0.2,0.7M,0.5M


## Packaging level — the escalation spectrum

In [3]:
pkg_labels = {1: "Official", 2: "Direct address", 3: "Produced", 4: "TikTok-native", 5: "Pop culture", 6: "Meme", 7: "Game UI"}

pkg = df.groupby("packaging_level").agg(
    count=("id", "size"),
    total_views=("views", "sum"),
    avg_views=("views", "mean"),
).sort_index()

pkg["label"] = pkg.index.map(pkg_labels)
pkg["pct_posts"] = (pkg["count"] / len(df) * 100).round(1)
pkg["pct_views"] = (pkg["total_views"] / df["views"].sum() * 100).round(1)
pkg["avg_views"] = pkg["avg_views"].apply(lambda x: f"{x/1e6:.1f}M")
pkg["total_views"] = pkg["total_views"].apply(lambda x: f"{x/1e6:.0f}M")

print("PACKAGING LEVELS — institutional to fictional")
print("="*70)
display(pkg[["label", "count", "pct_posts", "total_views", "pct_views", "avg_views"]])

# The algorithm's reward
meme_posts = df[df["is_meme"]]
non_meme = df[~df["is_meme"]]
print(f"\nLevel 5-7 posts: {len(meme_posts)} ({len(meme_posts)/len(df)*100:.0f}% of posts)")
print(f"Level 5-7 views: {meme_posts['views'].sum()/1e6:.0f}M ({meme_posts['views'].sum()/df['views'].sum()*100:.0f}% of views)")
print(f"Avg views L5-7: {meme_posts['views'].mean()/1e6:.1f}M vs L1-4: {non_meme['views'].mean()/1e6:.1f}M")

PACKAGING LEVELS — institutional to fictional


,label,count,pct_posts,total_views,pct_views,avg_views
packaging_level,,,,,,
1,Official,176,29.3,232M,16.0,1.3M
2,Direct address,15,2.5,32M,2.2,2.1M
3,Produced,211,35.2,406M,27.9,1.9M
4,TikTok-native,85,14.2,272M,18.7,3.2M
5,Pop culture,30,5.0,93M,6.4,3.1M
6,Meme,69,11.5,335M,23.1,4.9M
7,Game UI,14,2.3,83M,5.7,5.9M



Level 5-7 posts: 113 (19% of posts)
Level 5-7 views: 511M (35% of views)
Avg views L5-7: 4.5M vs L1-4: 1.9M


## Subject × Packaging — the matrix

In [4]:
# Count matrix
matrix_count = pd.crosstab(df["subject"], df["packaging_level"])
matrix_count.columns = [pkg_labels.get(c, c) for c in matrix_count.columns]
print("POST COUNT by Subject × Packaging")
display(matrix_count)

# Views matrix (average)
matrix_views = df.groupby(["subject", "packaging_level"])["views"].mean().unstack(fill_value=0)
matrix_views.columns = [pkg_labels.get(c, c) for c in matrix_views.columns]
print("\nAVERAGE VIEWS (millions) by Subject × Packaging")
display((matrix_views / 1e6).round(1))

POST COUNT by Subject × Packaging


,Official,Direct address,Produced,TikTok-native,Pop culture,Meme,Game UI
subject,,,,,,,
culture,20,1,34,6,4,2,1
enforcement,20,2,26,8,3,10,1
governance,79,4,58,6,1,0,1
opponents,22,0,8,13,8,32,0
promotion,3,7,34,24,8,6,1
trump,10,1,27,21,6,12,2
unclear,0,0,1,1,0,2,0
war,22,0,23,6,0,5,8



AVERAGE VIEWS (millions) by Subject × Packaging


,Official,Direct address,Produced,TikTok-native,Pop culture,Meme,Game UI
subject,,,,,,,
culture,1.9,1.5,2.5,1.7,2.3,5.7,0.4
enforcement,1.0,2.2,1.2,5.7,13.0,10.6,18.7
governance,1.1,3.0,1.6,5.7,2.9,0.0,2.9
opponents,1.0,0.0,0.8,1.6,1.9,2.1,0.0
promotion,0.6,1.9,1.4,2.7,1.2,1.8,1.7
trump,1.7,0.2,1.4,3.0,3.0,2.6,7.5
unclear,0.0,0.0,0.5,0.6,0.0,0.8,0.0
war,1.9,0.0,4.7,5.6,0.0,21.3,5.5


## War memes specifically

In [5]:
war = df[df["is_war"]]
war_meme = df[df["is_war_meme"]]
war_official = war[~war["is_meme"]]

print(f"War posts: {len(war)} ({len(war)/len(df)*100:.0f}% of all posts)")
print(f"War meme posts (L5-7): {len(war_meme)}")
print(f"War official posts (L1-4): {len(war_official)}")
print(f"\nWar meme views: {war_meme['views'].sum()/1e6:.0f}M ({war_meme['views'].sum()/df['views'].sum()*100:.0f}% of total)")
print(f"War official views: {war_official['views'].sum()/1e6:.0f}M")
print(f"Avg views — war meme: {war_meme['views'].mean()/1e6:.1f}M vs war official: {war_official['views'].mean()/1e6:.1f}M")

print(f"\nWar posts by packaging:")
for lvl in sorted(war["packaging_level"].unique()):
    subset = war[war["packaging_level"] == lvl]
    print(f"  L{lvl} {pkg_labels.get(lvl,''):15s}: {len(subset):3d} posts, {subset['views'].sum()/1e6:.0f}M views")

print(f"\nDate range of war posts: {war['date'].min().date()} to {war['date'].max().date()}")

# Game UI war posts — the peak memes
game_ui_war = war[war["packaging"] == "game_ui"]
print(f"\nGame UI war posts: {len(game_ui_war)}")
for _, p in game_ui_war.sort_values("date").iterrows():
    print(f"  {p['date'].date()}  {p['views']/1e6:.1f}M  {p['caption'][:50]}")

War posts: 64 (11% of all posts)
War meme posts (L5-7): 13
War official posts (L1-4): 51

War meme views: 150M (10% of total)
War official views: 184M
Avg views — war meme: 11.6M vs war official: 3.6M

War posts by packaging:
  L1 Official       :  22 posts, 43M views
  L3 Produced       :  23 posts, 108M views
  L4 TikTok-native  :   6 posts, 34M views
  L6 Meme           :   5 posts, 106M views
  L7 Game UI        :   8 posts, 44M views

Date range of war posts: 2025-09-02 to 2026-03-19

Game UI war posts: 8
  2026-03-04  9.8M  Iran can never have a nuclear weapon. Will continu
  2026-03-04  12.0M  Stay frosty 
  2026-03-06  3.9M  LOCKED IN ⭐️⭐️⭐️⭐️⭐️
  2026-03-06  4.0M  HOME RUN 🔥
  2026-03-06  2.7M  TOUCHDOWN ⚡️
  2026-03-06  4.1M  Justice the American way 
  2026-03-12  1.4M  UNDEFEATED. 🦅
  2026-03-12  6.3M  STRIKE 💥🦅


## War posts with meme packaging (L5-7) — scrollytelling step

In [6]:
# War posts with real military footage + meme packaging (L5-7)
# Used in the "war-memes-all" scrolly step
# Excludes FAFO/Maduro (no real_war_footage tag)

war_meme_strict = df[
    (df["subject"] == "war") & 
    (df["packaging_level"] >= 5) & 
    (df["tags"].apply(lambda t: "real_war_footage" in t if isinstance(t, list) else False))
].sort_values("date")

print(f"War posts with real footage + meme packaging (L5-7): {len(war_meme_strict)}")
print(f"Total views: {war_meme_strict['views'].sum():,.0f} ({war_meme_strict['views'].sum()/1e6:.1f}M)")
print(f"Avg views: {war_meme_strict['views'].mean()/1e6:.1f}M")
print(f"Date range: {war_meme_strict['date'].min().date()} to {war_meme_strict['date'].max().date()}")
print()
print("Posts (0-indexed for article.js):")
for _, p in war_meme_strict.iterrows():
    print(f"  #{p['post_num']:3d}  idx={p['post_num']-1:3d}  {p['date'].date()}  {p['views']/1e6:5.1f}M  L{p['packaging_level']}  {p['packaging']:12s}  {p['caption'][:40]}")

print()
print("JS array for article.js:")
indices = [str(p['post_num']-1) for _, p in war_meme_strict.iterrows()]
print(f"[{', '.join(indices)}]")

# Note: FAFO (#371) excluded -- classified as war/meme L6 but has no real_war_footage tag
# It is about the Maduro operation, not Iran war strikes

War posts with real footage + meme packaging (L5-7): 12
Total views: 69,800,000 (69.8M)
Avg views: 5.8M
Date range: 2026-03-04 to 2026-03-12

Posts (0-indexed for article.js):
  #516  idx=515  2026-03-04    9.8M  L7  game_ui       Iran can never have a nuclear weapon. Wi
  #518  idx=517  2026-03-04   12.0M  L7  game_ui       Stay frosty 
  #521  idx=520  2026-03-05    3.7M  L6  meme          “I was the hunted, now I’m the hunter”
  #522  idx=521  2026-03-05    2.0M  L6  meme          💥💥
  #523  idx=522  2026-03-05   12.9M  L6  meme          A message to terrorists: 
  #526  idx=525  2026-03-06    3.9M  L7  game_ui       LOCKED IN ⭐️⭐️⭐️⭐️⭐️
  #527  idx=526  2026-03-06    4.0M  L7  game_ui       HOME RUN 🔥
  #528  idx=527  2026-03-06    2.7M  L7  game_ui       TOUCHDOWN ⚡️
  #529  idx=528  2026-03-06    4.1M  L7  game_ui       Justice the American way 
  #538  idx=537  2026-03-09    7.0M  L6  meme          Coming in hot 🤫
  #544  idx=543  2026-03-12    1.4M  L7  game_ui       UNDEFEATED

## All 14 game_ui posts — the escalation endpoint

In [7]:
game_ui = df[df["packaging"] == "game_ui"].sort_values("date")
print(f"Game UI posts: {len(game_ui)} | Avg views: {game_ui['views'].mean()/1e6:.1f}M")
print(f"Subjects: {game_ui['subject'].value_counts().to_dict()}")
print()
for _, p in game_ui.iterrows():
    war_flag = " WAR" if p["subject"] == "war" else ""
    print(f"  #{p['post_num']:3d}  {p['date'].date()}  {p['views']/1e6:5.1f}M  {p['subject']:12s}{war_flag}  {p['caption'][:45]}")

Game UI posts: 14 | Avg views: 5.9M
Subjects: {'war': 8, 'trump': 2, 'enforcement': 1, 'governance': 1, 'promotion': 1, 'culture': 1}

  #128  2025-09-23   18.7M  enforcement   Gotta catch ‘em all  🎥: DHS
  #363  2025-12-30   14.5M  trump         crafting the GOLDEN AGE 🔥
  #380  2026-01-08    2.9M  governance    Out with the junk. In with REAL food. 
  #392  2026-01-13    1.7M  promotion     Say the Word on the Beat Challenge IMPOSSIBLE
  #440  2026-02-04    0.5M  trump         BEST RUN EVER 🏆
  #516  2026-03-04    9.8M  war          WAR  Iran can never have a nuclear weapon. Will co
  #518  2026-03-04   12.0M  war          WAR  Stay frosty 
  #526  2026-03-06    3.9M  war          WAR  LOCKED IN ⭐️⭐️⭐️⭐️⭐️
  #527  2026-03-06    4.0M  war          WAR  HOME RUN 🔥
  #528  2026-03-06    2.7M  war          WAR  TOUCHDOWN ⚡️
  #529  2026-03-06    4.1M  war          WAR  Justice the American way 
  #544  2026-03-12    1.4M  war          WAR  UNDEFEATED. 🦅
  #545  2026-03-12    6.3M  war   

In [8]:
fictional = df[df["tags"].apply(lambda t: "fictional_overlay" in t if isinstance(t, list) else False)].sort_values("date")
print(f"Fictional overlay posts: {len(fictional)} | Avg views: {fictional['views'].mean()/1e6:.1f}M")
print(f"Subjects: {fictional['subject'].value_counts().to_dict()}")
print(f"Packaging: {fictional['packaging'].value_counts().to_dict()}")
print()
for _, p in fictional.iterrows():
    war_flag = " WAR" if p["subject"] == "war" else ""
    print(f"  #{p['post_num']:3d}  {p['date'].date()}  {p['views']/1e6:5.1f}M  {p['packaging']:15s}  {p['subject']:12s}{war_flag}  {p['caption'][:50]}")

Fictional overlay posts: 53 | Avg views: 5.5M
Subjects: {'opponents': 17, 'war': 12, 'enforcement': 7, 'trump': 6, 'promotion': 5, 'culture': 3, 'governance': 2, 'unclear': 1}
Packaging: {'meme': 28, 'game_ui': 13, 'pop_culture': 8, 'tiktok': 3, 'produced': 1}

  #  7  2025-08-20    1.0M  pop_culture      promotion     All we do is win
  # 14  2025-08-21    2.5M  produced         promotion     START YOUR ENGINES 🇺🇸
  # 30  2025-08-26    1.0M  meme             opponents     Who the left swipes right on ➡️
  # 77  2025-09-09    0.7M  meme             enforcement   BORDER CZAR
  #128  2025-09-23   18.7M  game_ui          enforcement   Gotta catch ‘em all  🎥: DHS
  #164  2025-10-02    0.6M  meme             opponents     It’s not complicated 
  #167  2025-10-03    1.2M  pop_culture      opponents     It’s October 3rd 
  #174  2025-10-06    0.7M  meme             opponents     The longer the dems keep the gov shutdown the bigg
  #176  2025-10-07    0.7M  meme             opponents     Dems 

In [9]:
gamification = df[df["tags"].apply(lambda t: "gamification" in t if isinstance(t, list) else False)].sort_values("date")
print(f"Gamification posts: {len(gamification)} | Avg views: {gamification['views'].mean()/1e6:.1f}M")
print(f"Subjects: {gamification['subject'].value_counts().to_dict()}")
print(f"Packaging: {gamification['packaging'].value_counts().to_dict()}")
print()
for _, p in gamification.iterrows():
    war_flag = " WAR" if p["subject"] == "war" else ""
    fo = " [fictional_overlay]" if "fictional_overlay" in (p["tags"] or []) else ""
    print(f"  #{p['post_num']:3d}  {p['date'].date()}  {p['views']/1e6:5.1f}M  {p['packaging']:15s}  {p['subject']:12s}{war_flag}{fo}  {p['caption'][:45]}")

Gamification posts: 18 | Avg views: 5.2M
Subjects: {'war': 8, 'promotion': 3, 'opponents': 2, 'trump': 2, 'enforcement': 1, 'governance': 1, 'culture': 1}
Packaging: {'game_ui': 14, 'meme': 3, 'produced': 1}

  # 14  2025-08-21    2.5M  produced         promotion    [fictional_overlay]  START YOUR ENGINES 🇺🇸
  # 30  2025-08-26    1.0M  meme             opponents    [fictional_overlay]  Who the left swipes right on ➡️
  #121  2025-09-22    3.7M  meme             promotion     Eyetest👀
  #128  2025-09-23   18.7M  game_ui          enforcement  [fictional_overlay]  Gotta catch ‘em all  🎥: DHS
  #309  2025-12-04    2.6M  meme             opponents     They spread fake news. We listened (unfortuna
  #363  2025-12-30   14.5M  game_ui          trump        [fictional_overlay]  crafting the GOLDEN AGE 🔥
  #380  2026-01-08    2.9M  game_ui          governance   [fictional_overlay]  Out with the junk. In with REAL food. 
  #392  2026-01-13    1.7M  game_ui          promotion    [fictional_overlay

## Game lineage — subset options for grid highlight

Which posts to highlight on the 600-post grid when introducing the game lineage strand.
Comparing: `gamification` tag (non-war) vs `fictional_overlay` (non-war) vs pkg 5+ (non-war).

In [10]:
# Option 1: gamification tag, non-war
gam_nonwar = df[
    df["tags"].apply(lambda t: "gamification" in t if isinstance(t, list) else False)
    & (df["subject"] != "war")
].sort_values("date")

# Option 2: fictional_overlay tag, non-war
fic_nonwar = df[
    df["tags"].apply(lambda t: "fictional_overlay" in t if isinstance(t, list) else False)
    & (df["subject"] != "war")
].sort_values("date")

# Option 3: gamification OR fictional_overlay, non-war
either_nonwar = df[
    df["tags"].apply(lambda t: ("gamification" in t or "fictional_overlay" in t) if isinstance(t, list) else False)
    & (df["subject"] != "war")
].sort_values("date")

print(f"Option 1 — gamification, non-war: {len(gam_nonwar)} posts")
print(f"Option 2 — fictional_overlay, non-war: {len(fic_nonwar)} posts")
print(f"Option 3 — gamification OR fictional_overlay, non-war: {len(either_nonwar)} posts")
print()

print("=" * 60)
print("OPTION 1: gamification, non-war")
print("=" * 60)
for _, p in gam_nonwar.iterrows():
    print(f"  #{p['post_num']:3d}  {p['date'].date()}  {p['views']/1e6:5.1f}M  {p['subject']:12s}  {p['caption'][:50]}")

print()
print("Grid indices (0-indexed) for JS:")
print([int(p['post_num'] - 1) for _, p in gam_nonwar.iterrows()])


Option 1 — gamification, non-war: 10 posts
Option 2 — fictional_overlay, non-war: 41 posts
Option 3 — gamification OR fictional_overlay, non-war: 44 posts

OPTION 1: gamification, non-war
  # 14  2025-08-21    2.5M  promotion     START YOUR ENGINES 🇺🇸
  # 30  2025-08-26    1.0M  opponents     Who the left swipes right on ➡️
  #121  2025-09-22    3.7M  promotion     Eyetest👀
  #128  2025-09-23   18.7M  enforcement   Gotta catch ‘em all  🎥: DHS
  #309  2025-12-04    2.6M  opponents     They spread fake news. We listened (unfortunately)
  #363  2025-12-30   14.5M  trump         crafting the GOLDEN AGE 🔥
  #380  2026-01-08    2.9M  governance    Out with the junk. In with REAL food. 
  #392  2026-01-13    1.7M  promotion     Say the Word on the Beat Challenge IMPOSSIBLE! 🇺🇸 
  #440  2026-02-04    0.5M  trump         BEST RUN EVER 🏆
  #585  2026-03-26    0.4M  culture       OPENING DAY VIBES 🦅🇺🇸⚾️

Grid indices (0-indexed) for JS:
[13, 29, 120, 127, 308, 362, 379, 391, 439, 584]


## Tag frequency — what techniques appear most

In [11]:
tag_counts = tag_rows["tag"].value_counts()
tag_pct = (tag_counts / len(df) * 100).round(1)

# Also compute avg views per tag
tag_views = tag_rows.groupby("tag")["views"].mean().sort_values(ascending=False)

tag_summary = pd.DataFrame({
    "count": tag_counts,
    "pct_posts": tag_pct,
    "avg_views": (tag_views / 1e6).round(1),
}).sort_values("count", ascending=False)

print("TAG FREQUENCY (top 25)")
print("="*60)
display(tag_summary.head(25))

print(f"\nTags that correlate with HIGH views (avg > 3M):")
high_view_tags = tag_summary[tag_summary["avg_views"] > 3.0].sort_values("avg_views", ascending=False)
display(high_view_tags)

TAG FREQUENCY (top 25)


,count,pct_posts,avg_views
tag,,,
speech_audio,360,60.0,1.8
highlight_reel,299,49.8,3.1
aura_farming,210,35.0,3.1
dramatic_score,174,29.0,1.8
speed_ramp,169,28.2,3.5
stock_hero,154,25.7,4.4
provocative,139,23.2,4.6
trending_audio,107,17.8,4.9
troll,104,17.3,4.9



Tags that correlate with HIGH views (avg > 3M):


,count,pct_posts,avg_views
tag,,,
expletive,35,5.8,7.1
greyscale_compare,5,0.8,7.0
real_war_footage,29,4.8,6.1
hyper_masculine,77,12.8,6.1
fictional_overlay,53,8.8,5.5
bass_drop,99,16.5,5.4
gamification,18,3.0,5.2
punchline,81,13.5,5.1
trending_audio,107,17.8,4.9


## Monthly posting pace and packaging escalation

In [12]:
monthly = df.groupby("month").agg(
    posts=("id", "size"),
    total_views=("views", "sum"),
    avg_pkg=("packaging_level", "mean"),
    meme_count=("is_meme", "sum"),
    war_count=("is_war", "sum"),
    war_meme_count=("is_war_meme", "sum"),
)
monthly["meme_pct"] = (monthly["meme_count"] / monthly["posts"] * 100).round(0)
monthly["avg_views"] = (monthly["total_views"] / monthly["posts"] / 1e6).round(1)
monthly["avg_pkg"] = monthly["avg_pkg"].round(2)

print("MONTHLY BREAKDOWN")
print("="*90)
display(monthly[["posts", "avg_views", "avg_pkg", "meme_count", "meme_pct", "war_count", "war_meme_count"]])

MONTHLY BREAKDOWN


,posts,avg_views,avg_pkg,meme_count,meme_pct,war_count,war_meme_count
month,,,,,,,
2025-08,48,1.0,2.81,6,12.0,0,0
2025-09,110,1.2,2.54,13,12.0,3,0
2025-10,82,1.5,3.23,24,29.0,6,0
2025-11,62,3.2,3.40,12,19.0,3,0
2025-12,63,4.7,3.49,16,25.0,3,0
2026-01,71,4.3,3.17,11,15.0,7,1
2026-02,71,1.9,2.89,9,13.0,8,0
2026-03,92,2.4,3.24,22,24.0,34,12
2026-04,1,0.2,3.00,0,0.0,0,0


## The silence — before and after March 16

In [13]:
before = df[(df["date"] >= "2026-03-01") & (df["date"] <= "2026-03-16")]
after = df[df["date"] > "2026-03-16"]

print(f"Mar 1-16: {len(before)} posts, avg views {before['views'].mean()/1e6:.1f}M")
print(f"  War posts: {before['is_war'].sum()}, War memes: {before['is_war_meme'].sum()}")
print(f"  Avg packaging level: {before['packaging_level'].mean():.1f}")

print(f"\nAfter Mar 16: {len(after)} posts, avg views {after['views'].mean()/1e6:.1f}M")
print(f"  War posts: {after['is_war'].sum()}, War memes: {after['is_war_meme'].sum()}")
print(f"  Avg packaging level: {after['packaging_level'].mean():.1f}")
print(f"  Subjects: {after['subject'].value_counts().to_dict()}")

if before['views'].mean() > 0:
    drop = (1 - after['views'].mean() / before['views'].mean()) * 100
    print(f"\nViews drop: {drop:.0f}%")

Mar 1-16: 50 posts, avg views 3.6M
  War posts: 32, War memes: 12
  Avg packaging level: 3.2

After Mar 16: 43 posts, avg views 1.0M
  War posts: 2, War memes: 0
  Avg packaging level: 3.3
  Subjects: {'governance': 9, 'culture': 8, 'opponents': 7, 'promotion': 7, 'enforcement': 6, 'war': 2, 'trump': 2, 'unclear': 2}

Views drop: 73%


## Tag co-occurrence — what techniques cluster together

In [14]:
# Build a tag presence matrix (one-hot)
all_tags = sorted(tag_rows["tag"].dropna().unique())
for t in all_tags:
    df[f"t_{t}"] = df["tags"].apply(lambda tags: t in tags if isinstance(tags, list) else False)

tag_cols = [c for c in df.columns if c.startswith("t_")]

# Co-occurrence: for key audience-signal tags, what else appears with them
signal_tags = ["gamification", "fictional_overlay", "troll", "meme_audio", "hyper_masculine", 
               "aura_farming", "provocative", "real_war_footage", "trending_audio", "bass_drop"]

print("TAG CO-OCCURRENCE — what appears alongside each signal tag")
print("="*70)
for st in signal_tags:
    col = f"t_{st}"
    if col not in df.columns:
        continue
    subset = df[df[col]]
    if len(subset) == 0:
        continue
    # Find top co-occurring tags
    cooc = {}
    for tc in tag_cols:
        other_tag = tc[2:]
        if other_tag == st:
            continue
        cooc[other_tag] = subset[tc].sum()
    top = sorted(cooc.items(), key=lambda x: -x[1])[:8]
    top_str = ", ".join(f"{t}({n})" for t, n in top)
    print(f"  {st} ({len(subset)} posts): {top_str}")

# Clean up temp columns
df.drop(columns=tag_cols, inplace=True)

TAG CO-OCCURRENCE — what appears alongside each signal tag
  gamification (18 posts): fictional_overlay(15), provocative(14), troll(14), meme_audio(13), highlight_reel(10), hyper_masculine(10), punchline(10), real_war_footage(10)
  fictional_overlay (53 posts): troll(43), meme_audio(42), provocative(39), highlight_reel(29), punchline(28), stock_hero(25), named_target(22), bass_drop(20)
  troll (104 posts): provocative(78), meme_audio(67), named_target(56), punchline(55), highlight_reel(53), fictional_overlay(43), speech_audio(36), stock_hero(34)
  meme_audio (103 posts): troll(67), provocative(64), highlight_reel(53), punchline(46), fictional_overlay(42), named_target(39), bass_drop(38), stock_hero(35)
  hyper_masculine (77 posts): highlight_reel(64), aura_farming(51), stock_hero(51), speed_ramp(39), bass_drop(38), provocative(38), speech_audio(32), dramatic_score(31)
  aura_farming (210 posts): highlight_reel(159), speed_ramp(127), speech_audio(102), stock_hero(99), dramatic_score(91)

## Top posts by views

In [15]:
top20 = df.nlargest(20, "views")[["post_num", "date", "caption", "views", "subject", "packaging", "packaging_level"]].copy()
top20["views"] = top20["views"].apply(lambda x: f"{x/1e6:.1f}M")
top20["date"] = top20["date"].dt.date
print("TOP 20 POSTS BY VIEWS")
print("="*90)
display(top20.to_string(index=False))

TOP 20 POSTS BY VIEWS


' post_num       date                                                                      caption views     subject   packaging  packaging_level\n      371 2026-01-03                                                                       FAFO.  80.7M         war        meme                6\n      323 2025-12-10 All aboard the Deportation Express! Next stop: Back to where you came from ✨ 38.0M enforcement        meme                6\n      314 2025-12-05        PSA: If you’re a criminal illegal, you WILL be arrested & deported. ✨ 25.3M enforcement        meme                6\n      534 2026-03-09                                             Can’t say he didn’t warn them. 🦅 25.2M         war    produced                3\n      393 2026-01-14                                                         AMERICAN ENERGY 🇺🇸⚡️ 23.6M  governance      tiktok                4\n      387 2026-01-11                                                           Border is LOCKED 🔒 22.9M enforcement pop_cul

## Packaging escalation over time — rolling average

In [16]:
# Rolling 20-post average of packaging level over time
df_sorted = df.sort_values("date").reset_index(drop=True)
df_sorted["pkg_rolling"] = df_sorted["packaging_level"].rolling(20, center=True).mean()
df_sorted["views_rolling"] = df_sorted["views"].rolling(20, center=True).mean()

# Print as a simple timeline
print("PACKAGING ESCALATION — 20-post rolling average")
print("="*60)
checkpoints = [0, 100, 200, 300, 400, 500, len(df_sorted)-1]
for i in checkpoints:
    row = df_sorted.iloc[i]
    print(f"  Post #{i+1:3d}  {row['date'].date()}  avg pkg: {row['pkg_rolling']:.2f}  avg views: {row['views_rolling']/1e6:.1f}M")

PACKAGING ESCALATION — 20-post rolling average
  Post #  1  2025-08-19  avg pkg: nan  avg views: nanM
  Post #101  2025-09-16  avg pkg: 2.15  avg views: 0.5M
  Post #201  2025-10-17  avg pkg: 2.55  avg views: 1.3M
  Post #301  2025-11-27  avg pkg: 4.00  avg views: 5.2M
  Post #401  2026-01-16  avg pkg: 3.00  avg views: 2.7M
  Post #501  2026-02-27  avg pkg: 2.30  avg views: 2.9M
  Post #600  2026-04-01  avg pkg: nan  avg views: nanM


## Expletive / profanity posts

In [17]:
expletive = df[df["tags"].apply(lambda t: "expletive" in t if isinstance(t, list) else False)]
print(f"Posts tagged 'expletive': {len(expletive)}")
print(f"Avg views: {expletive['views'].mean()/1e6:.1f}M vs non-expletive: {df[~df.index.isin(expletive.index)]['views'].mean()/1e6:.1f}M")
print()
for _, p in expletive.sort_values("date").iterrows():
    print(f"  #{p['post_num']:3d}  {p['date'].date()}  {p['views']/1e6:5.1f}M  {p['subject']:12s} L{p['packaging_level']}  {p['caption'][:45]}")

Posts tagged 'expletive': 35
Avg views: 7.1M vs non-expletive: 2.1M

  #  7  2025-08-20    1.0M  promotion    L5  All we do is win
  # 50  2025-09-01    0.2M  promotion    L5  
  # 96  2025-09-15    0.4M  war          L1  FAFO. 
  #123  2025-09-22    1.0M  promotion    L6  For Charlie 
  #134  2025-09-24    0.6M  opponents    L1  VP Vance speaking the truth🔥 
  #174  2025-10-06    0.7M  opponents    L6  The longer the dems keep the gov shutdown the
  #205  2025-10-17    1.0M  opponents    L1  FAFO. 
  #211  2025-10-20    2.5M  promotion    L3  2 Million followers - LFG!!!
  #210  2025-10-20    1.2M  governance   L1  Still our king 🇺🇸 MAGA. 
  #221  2025-10-24    1.5M  trump        L3  Can’t stop winning 🔥
  #225  2025-10-27    3.6M  opponents    L5  Democrats are making this WAY harder than it 
  #251  2025-11-06    3.6M  trump        L6  One year since the lib MELTDOWN 
  #322  2025-12-09    8.5M  culture      L6  Then vs now. 😤
  #336  2025-12-16    1.7M  trump        L4  And that’s 

## The "strongman trifecta" — war + enforcement + trump

In [18]:
trifecta_subjects = ["war", "enforcement", "trump"]
trifecta = df[df["subject"].isin(trifecta_subjects)]
rest = df[~df["subject"].isin(trifecta_subjects)]

print(f"Strongman trifecta (war + enforcement + trump):")
print(f"  Posts: {len(trifecta)} ({len(trifecta)/len(df)*100:.1f}%)")
print(f"  Views: {trifecta['views'].sum()/1e6:.0f}M ({trifecta['views'].sum()/df['views'].sum()*100:.1f}%)")
print(f"  Avg views: {trifecta['views'].mean()/1e6:.1f}M")
print(f"\nEverything else (governance + opponents + culture + promotion):")
print(f"  Posts: {len(rest)} ({len(rest)/len(df)*100:.1f}%)")
print(f"  Views: {rest['views'].sum()/1e6:.0f}M ({rest['views'].sum()/df['views'].sum()*100:.1f}%)")
print(f"  Avg views: {rest['views'].mean()/1e6:.1f}M")

Strongman trifecta (war + enforcement + trump):
  Posts: 213 (35.5%)
  Views: 783M (53.9%)
  Avg views: 3.7M

Everything else (governance + opponents + culture + promotion):
  Posts: 387 (64.5%)
  Views: 670M (46.1%)
  Avg views: 1.7M


## Boundary-testing posts — candidates for lineage ancestors

Posts that pushed a specific boundary early on, before the war.

In [19]:
pre_war = df[df["date"] < "2026-03-01"]

# Posts with high packaging on non-war content — the testing ground
boundary = pre_war[pre_war["packaging_level"] >= 5].sort_values("date")
print(f"Pre-war posts at L5-7 (boundary-testing): {len(boundary)}")
print("="*80)
for _, p in boundary.iterrows():
    tags_str = ", ".join(p["tags"][:5]) if isinstance(p["tags"], list) else ""
    print(f"  #{p['post_num']:3d}  {p['date'].date()}  {p['views']/1e6:5.1f}M  {p['subject']:12s} {p['packaging']:15s} L{p['packaging_level']}  {p['caption'][:35]}")
    print(f"         {tags_str}")
    print()

Pre-war posts at L5-7 (boundary-testing): 91
  #  3  2025-08-20    1.9M  trump        meme            L6  ‘I was the hunted, and now I’m the 
         meme_audio, stock_hero, comparison, highlight_reel, punchline

  #  5  2025-08-20    1.6M  trump        meme            L6  They don’t know it yet…
         meme_audio, speed_ramp, stock_hero, comparison, highlight_reel

  #  7  2025-08-20    1.0M  promotion    pop_culture     L5  All we do is win
         meme_audio, hype_language, deep_fried, stock_hero, highlight_reel

  # 13  2025-08-21    0.8M  opponents    meme            L6  Fake news
         meme_audio, bass_drop, speech_audio, highlight_reel, punchline

  # 24  2025-08-24    1.4M  opponents    meme            L6  ASMR
         meme_audio, punchline, provocative

  # 30  2025-08-26    1.0M  opponents    meme            L6  Who the left swipes right on ➡️
         trending_audio, gamification, fictional_overlay, punchline, named_target

  # 50  2025-09-01    0.2M  promotion    po

## Real military footage — video review

All posts tagged `real_war_footage`, with embedded videos and thumbnails for visual review.

In [20]:
war_footage = df[df["tags"].apply(lambda t: "real_war_footage" in t if isinstance(t, list) else False)].sort_values("date")
print(f"{len(war_footage)} posts with real military footage\n")

# Build HTML grid with thumbnails and video embeds
html = '<div style="display:flex;flex-wrap:wrap;gap:16px;">'
for _, row in war_footage.iterrows():
    vid_path = PEAK_MEME / "videos" / f"{row['id']}.mp4"
    img_path = PEAK_MEME / "images" / "grid" / f"{row['id']}.jpg"
    pkg = row["packaging_level"]
    tier_color = "#555" if pkg <= 2 else ("#4a9eff" if pkg <= 4 else "#ff3b3b")
    tier_label = "Plain" if pkg <= 2 else ("Produced" if pkg <= 4 else "Meme-packaged")
    views = f"{row['views']/1e6:.1f}M" if row['views'] >= 1e6 else f"{row['views']/1e3:.0f}K"
    caption = (row['caption'] or '')[:60]
    html += f'''
    <div style="width:180px;background:#111;border-radius:8px;overflow:hidden;border:2px solid {tier_color};">
        <video src="{vid_path}" width="180" controls muted preload="none" poster="{img_path}"
               style="display:block;aspect-ratio:9/16;object-fit:cover;"></video>
        <div style="padding:6px 8px;font-size:11px;color:#ccc;">
            <div style="font-weight:700;color:#fff;">#{row['post_num']}</div>
            <div>{row['date'].strftime('%Y-%m-%d')}</div>
            <div style="color:{tier_color};font-weight:600;">{tier_label} (L{pkg})</div>
            <div>{views}</div>
            <div style="color:#888;margin-top:2px;">{caption}</div>
        </div>
    </div>'''
html += '</div>'
display(HTML(html))

29 posts with real military footage



## Gamification posts (non-war) — video review

All posts with the `gamification` tag on non-war subjects. These are the posts where game-UI overlays were tested before being applied to combat footage.

In [21]:
gam_nonwar = df[
    df["tags"].apply(lambda t: "gamification" in t if isinstance(t, list) else False)
    & (df["subject"] != "war")
].sort_values("date")
print(f"{len(gam_nonwar)} gamification posts on non-war subjects\n")
print(f"Subjects: {gam_nonwar['subject'].value_counts().to_dict()}")
print(f"Grid indices (0-indexed): {[int(p['post_num'] - 1) for _, p in gam_nonwar.iterrows()]}\n")

# Build HTML grid with thumbnails and video embeds
html = '<div style="display:flex;flex-wrap:wrap;gap:16px;">'
for _, row in gam_nonwar.iterrows():
    vid_path = PEAK_MEME / "videos" / f"{row['id']}.mp4"
    img_path = PEAK_MEME / "images" / "grid" / f"{row['id']}.jpg"
    views = f"{row['views']/1e6:.1f}M" if row['views'] >= 1e6 else f"{row['views']/1e3:.0f}K"
    caption = (row['caption'] or '')[:60]
    html += f'''
    <div style="width:180px;background:#111;border-radius:8px;overflow:hidden;border:2px solid #f59e0b;">
        <video src="{vid_path}" width="180" controls muted preload="none" poster="{img_path}"
               style="display:block;aspect-ratio:9/16;object-fit:cover;"></video>
        <div style="padding:6px 8px;font-size:11px;color:#ccc;">
            <div style="font-weight:700;color:#fff;">#{row['post_num']} — {row['subject']}</div>
            <div>{row['date'].strftime('%Y-%m-%d')}</div>
            <div style="color:#f59e0b;font-weight:600;">L{row['packaging_level']}</div>
            <div>{views}</div>
            <div style="color:#888;margin-top:2px;">{caption}</div>
        </div>
    </div>'''
html += '</div>'
display(HTML(html))

10 gamification posts on non-war subjects

Subjects: {'promotion': 3, 'opponents': 2, 'trump': 2, 'enforcement': 1, 'governance': 1, 'culture': 1}
Grid indices (0-indexed): [13, 29, 120, 127, 308, 362, 379, 391, 439, 584]



## Fictional overlay posts (non-war, pre-war) — video review

All posts with the `fictional_overlay` tag on non-war subjects, before the war began (Mar 1, 2026). These are posts where fiction was composited onto real content.

In [22]:
fic_nonwar = df[
    df["tags"].apply(lambda t: "fictional_overlay" in t if isinstance(t, list) else False)
    & (df["subject"] != "war")
    & (df["date"] < "2026-03-01")
].sort_values("date")
print(f"{len(fic_nonwar)} fictional_overlay posts on non-war subjects (pre-war)\n")
print(f"Subjects: {fic_nonwar['subject'].value_counts().to_dict()}")
print(f"Grid indices (0-indexed): {[int(p['post_num'] - 1) for _, p in fic_nonwar.iterrows()]}\n")

# Build HTML grid with thumbnails and video embeds
html = '<div style="display:flex;flex-wrap:wrap;gap:16px;">'
for _, row in fic_nonwar.iterrows():
    vid_path = PEAK_MEME / "videos" / f"{row['id']}.mp4"
    img_path = PEAK_MEME / "images" / "grid" / f"{row['id']}.jpg"
    views = f"{row['views']/1e6:.1f}M" if row['views'] >= 1e6 else f"{row['views']/1e3:.0f}K"
    caption = (row['caption'] or '')[:60]
    html += f'''
    <div style="width:180px;background:#111;border-radius:8px;overflow:hidden;border:2px solid #a855f7;">
        <video src="{vid_path}" width="180" controls muted preload="none" poster="{img_path}"
               style="display:block;aspect-ratio:9/16;object-fit:cover;"></video>
        <div style="padding:6px 8px;font-size:11px;color:#ccc;">
            <div style="font-weight:700;color:#fff;">#{row['post_num']} — {row['subject']}</div>
            <div>{row['date'].strftime('%Y-%m-%d')}</div>
            <div style="color:#a855f7;font-weight:600;">L{row['packaging_level']}</div>
            <div>{views}</div>
            <div style="color:#888;margin-top:2px;">{caption}</div>
        </div>
    </div>'''
html += '</div>'
display(HTML(html))

39 fictional_overlay posts on non-war subjects (pre-war)

Subjects: {'opponents': 16, 'enforcement': 7, 'trump': 6, 'promotion': 5, 'culture': 3, 'unclear': 1, 'governance': 1}
Grid indices (0-indexed): [6, 13, 29, 76, 127, 163, 166, 173, 175, 185, 200, 218, 219, 228, 232, 234, 236, 240, 256, 269, 294, 313, 315, 321, 322, 334, 341, 343, 363, 362, 375, 377, 379, 386, 391, 419, 424, 439, 500]



## Profanity sources breakdown

Categorising where the profanities came from across the 35 posts.

In [23]:
import json
from collections import Counter

with open(PEAK_MEME / 'data' / 'profanity.json') as f:
    prof = json.load(f)

# Categorise each post's source
def categorise(source):
    s = source.lower()
    if 'trump' in s:
        return 'Trump direct speech'
    if any(w in s for w in ['rap', 'song', 'music', 'cent', 'drake', 'cardi', 'victory lap', 'johnson', 'chalamet']):
        return 'Rap'
    if 'pitbull' in s:
        return 'Pitbull'
    if 'vance' in s or 'jd' in s:
        return 'JD Vance'
    if 'hegseth' in s or 'pete h' in s:
        return 'Pete Hegseth'
    if 'ai-' in s or 'ai ' in s:
        return 'AI-generated'
    if 'protestor' in s or 'protest' in s:
        return 'Other'
    if any(w in s for w in ['movie', 'clip', 'brothers', 'hangover', 'wolf', 'deadpool', 'gta', 'rick']):
        return 'Meme clips'
    if any(w in s for w in ['caption', 'text overlay', 'acronym']):
        return 'Text & acronyms (if not already sorted)'
    return 'Other'

cats = Counter()
details = {}
for p in prof['posts']:
    cat = categorise(p['source'])
    cats[cat] += 1
    details.setdefault(cat, []).append(f"  {p['date']} | {p['source']} | {p.get('caption','')[:40]}")

print('=== Profanity source categories ===')
for cat, count in cats.most_common():
    print(f'{cat}: {count}')
    for d in details[cat]:
        print(d)
    print()

# Treatment breakdown
treatments = Counter(p['treatment'] for p in prof['posts'])
print('=== Treatment breakdown ===')
for t, c in treatments.most_common():
    print(f'{t}: {c}')

print(f"\nTotal: {len(prof['posts'])} posts")
print(f"Not on Instagram: {sum(1 for p in prof['posts'] if p['on_instagram'] == 'no')}")
print(f"Scrubbed for Instagram: {sum(1 for p in prof['posts'] if p['on_instagram'] == 'yes_scrubbed')}")


=== Profanity source categories ===
Meme clips: 8
  2025-08-20 | Movie/meme clip | All we do is win
  2025-09-01 | Step Brothers clip | TikTok video #7545165161884028174
  2025-09-22 | The Hangover audio clip | For Charlie 
  2025-10-27 | Wolf of Wall Street + Rick & Morty clips | Democrats are making this WAY harder tha
  2026-01-11 | Movie clip | Border is LOCKED 🔒
  2026-01-24 | Deadpool & Wolverine clip | ONE YEAR OF MAGA… Kind of a big deal 
  2026-03-06 | GTA: San Andreas clip | LOCKED IN ⭐️⭐️⭐️⭐️⭐️
  2026-03-18 | Wolf of Wall Street clip | $10.5 TRILLION IN 1 YEAR 🤑

Text & acronyms (if not already sorted): 7
  2025-09-15 | Caption/acronym (FAFO) | FAFO. 
  2025-10-20 | Caption/acronym (LFG) | 2 Million followers - LFG!!!
  2025-12-31 | Caption/acronym (LFG) | THE YEAR AMERICA LEVELED UP 🔥 LFG 
  2026-01-03 | Caption (FAFO) | FAFO. 
  2026-01-05 | Caption LFG + bleeped voiceover | LFG!! 🦅
  2026-01-19 | Text overlay + voiceover | Everybody wants the view but no one want
  2026-0

## War meme views — disproportionate impact

How meme-packaged war posts compare to the rest of the account.

In [24]:
total = len(df)
total_views = df['views'].sum()

# War meme posts (war subject + packaging level 5+)
war_memes = df[(df['subject'] == 'war') & (df['packaging_level'] >= 5)]
wm_count = len(war_memes)
wm_views = war_memes['views'].sum()

# All war posts
war_all = df[df['subject'] == 'war']

# All meme-packaged
meme_all = df[df['packaging_level'] >= 5]

# Rest (non-war-meme)
rest = df[~((df['subject'] == 'war') & (df['packaging_level'] >= 5))]
rest_avg = rest['views'].mean()

print('=== War meme posts vs rest ===')
print(f'Total: {total} posts, {total_views/1e9:.2f}B views')
print(f'War memes: {wm_count} posts ({wm_count/total*100:.1f}% of posts)')
print(f'War meme views: {wm_views/1e6:.0f}M ({wm_views/total_views*100:.1f}% of all views)')
print(f'War meme avg: {wm_views/wm_count/1e6:.1f}M views per post')
print(f'Rest avg: {rest_avg/1e6:.1f}M views per post')
print(f'Ratio: {(wm_views/wm_count)/rest_avg:.1f}x')
print()
print('=== All war posts ===')
print(f'{len(war_all)} posts ({len(war_all)/total*100:.1f}%), {war_all["views"].sum()/1e6:.0f}M views ({war_all["views"].sum()/total_views*100:.1f}%)')
print()
print('=== All meme-packaged ===')
print(f'{len(meme_all)} posts ({len(meme_all)/total*100:.1f}%), {meme_all["views"].sum()/1e6:.0f}M views ({meme_all["views"].sum()/total_views*100:.1f}%)')
print()
print('=== Individual war meme posts ===')
for _, p in war_memes.sort_values('views', ascending=False).iterrows():
    print(f'  #{p["post_num"]:3d} | {p["date"].date()} | {p["views"]/1e6:.1f}M | L{p["packaging_level"]} | {p["caption"][:45]}')


=== War meme posts vs rest ===
Total: 600 posts, 1.45B views
War memes: 13 posts (2.2% of posts)
War meme views: 150M (10.4% of all views)
War meme avg: 11.6M views per post
Rest avg: 2.2M views per post
Ratio: 5.2x

=== All war posts ===
64 posts (10.7%), 335M views (23.0%)

=== All meme-packaged ===
113 posts (18.8%), 511M views (35.2%)

=== Individual war meme posts ===
  #371 | 2026-01-03 | 80.7M | L6 | FAFO. 
  #523 | 2026-03-05 | 12.9M | L6 | A message to terrorists: 
  #518 | 2026-03-04 | 12.0M | L7 | Stay frosty 
  #516 | 2026-03-04 | 9.8M | L7 | Iran can never have a nuclear weapon. Will co
  #538 | 2026-03-09 | 7.0M | L6 | Coming in hot 🤫
  #545 | 2026-03-12 | 6.3M | L7 | STRIKE 💥🦅
  #529 | 2026-03-06 | 4.1M | L7 | Justice the American way 
  #527 | 2026-03-06 | 4.0M | L7 | HOME RUN 🔥
  #526 | 2026-03-06 | 3.9M | L7 | LOCKED IN ⭐️⭐️⭐️⭐️⭐️
  #521 | 2026-03-05 | 3.7M | L6 | “I was the hunted, now I’m the hunter”
  #528 | 2026-03-06 | 2.7M | L7 | TOUCHDOWN ⚡️
  #522 | 2026-03-05

## Iran war memes — closing finding

Comparing meme-packaged (L5-7) vs non-meme (L1-4) Iran war posts on views and engagement.

In [25]:
import csv

# Load engagement data
engagement = {}
with open(WAR_MEMES / 'data' / 'wh_tiktok_complete.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        vid = row.get('video_id', '')
        try:
            engagement[vid] = {
                'likes': int(float(row.get('like_count', 0) or 0)),
                'comments': int(float(row.get('comment_count', 0) or 0))
            }
        except: pass

# Iran war memes (war subject, pkg 5+, after Feb 28)
iran_war_memes = df[
    (df['subject'] == 'war') &
    (df['packaging_level'] >= 5) &
    (df['date'] >= '2026-02-28')
]

# Iran war non-meme (war subject, pkg < 5, after Feb 28)
iran_war_plain = df[
    (df['subject'] == 'war') &
    (df['packaging_level'] < 5) &
    (df['date'] >= '2026-02-28')
]

# All account
total_posts = len(df)
total_views = df['views'].sum()

def get_engagement(subset):
    likes = sum(engagement.get(row['id'], {}).get('likes', 0) for _, row in subset.iterrows())
    comments = sum(engagement.get(row['id'], {}).get('comments', 0) for _, row in subset.iterrows())
    return likes, comments

wm_likes, wm_comments = get_engagement(iran_war_memes)
wp_likes, wp_comments = get_engagement(iran_war_plain)
all_likes, all_comments = sum(engagement.get(row['id'], {}).get('likes', 0) for _, row in df.iterrows()), sum(engagement.get(row['id'], {}).get('comments', 0) for _, row in df.iterrows())

print('=== Iran war memes (L5-7, after Feb 28) ===')
print(f'Posts: {len(iran_war_memes)}')
print(f'Views: {iran_war_memes["views"].sum()/1e6:.1f}M')
print(f'Avg views: {iran_war_memes["views"].mean()/1e6:.1f}M')
print(f'Likes: {wm_likes:,} | Comments: {wm_comments:,}')
print(f'Like/comment ratio: {wm_likes/wm_comments:.0f}:1')
print(f'Avg likes: {wm_likes/len(iran_war_memes)/1e3:.1f}K')
print(f'Avg comments: {wm_comments/len(iran_war_memes)/1e3:.1f}K')
print()
print('=== Iran war non-meme (L1-4, after Feb 28) ===')
print(f'Posts: {len(iran_war_plain)}')
print(f'Views: {iran_war_plain["views"].sum()/1e6:.1f}M')
print(f'Avg views: {iran_war_plain["views"].mean()/1e6:.1f}M')
print(f'Likes: {wp_likes:,} | Comments: {wp_comments:,}')
print(f'Like/comment ratio: {wp_likes/wp_comments:.0f}:1')
print()
print('=== Account-wide ===')
print(f'Total: {total_posts} posts, {total_views/1e9:.2f}B views')
print(f'Likes: {all_likes:,} | Comments: {all_comments:,}')
print(f'Like/comment ratio: {all_likes/all_comments:.0f}:1')
print(f'Avg views: {total_views/total_posts/1e6:.1f}M')
print()
print('=== Comparison ===')
print(f'Iran war memes vs non-meme war:')
print(f'  Views: {iran_war_memes["views"].mean()/iran_war_plain["views"].mean():.1f}x more per post')
print(f'  Like/comment: {wm_likes/wm_comments:.0f}:1 vs {wp_likes/wp_comments:.0f}:1')
print()
print('=== Individual Iran war memes ===')
for _, p in iran_war_memes.sort_values('views', ascending=False).iterrows():
    e = engagement.get(p['id'], {})
    lc = e.get('likes',0) / e.get('comments',1) if e.get('comments',0) > 0 else 0
    print(f'  #{p["post_num"]:3d} | {p["date"].date()} | {p["views"]/1e6:.1f}M | L{p["packaging_level"]} | L/C {lc:.0f}:1 | {p["caption"][:40]}')


=== Iran war memes (L5-7, after Feb 28) ===
Posts: 12
Views: 69.8M
Avg views: 5.8M
Likes: 6,003,800 | Comments: 74,865
Like/comment ratio: 80:1
Avg likes: 500.3K
Avg comments: 6.2K

=== Iran war non-meme (L1-4, after Feb 28) ===
Posts: 25
Views: 95.4M
Avg views: 3.8M
Likes: 8,705,100 | Comments: 119,709
Like/comment ratio: 73:1

=== Account-wide ===
Total: 600 posts, 1.45B views
Likes: 113,688,860 | Comments: 2,831,811
Like/comment ratio: 40:1
Avg views: 2.4M

=== Comparison ===
Iran war memes vs non-meme war:
  Views: 1.5x more per post
  Like/comment: 80:1 vs 73:1

=== Individual Iran war memes ===
  #523 | 2026-03-05 | 12.9M | L6 | L/C 132:1 | A message to terrorists: 
  #518 | 2026-03-04 | 12.0M | L7 | L/C 94:1 | Stay frosty 
  #516 | 2026-03-04 | 9.8M | L7 | L/C 90:1 | Iran can never have a nuclear weapon. Wi
  #538 | 2026-03-09 | 7.0M | L6 | L/C 48:1 | Coming in hot 🤫
  #545 | 2026-03-12 | 6.3M | L7 | L/C 153:1 | STRIKE 💥🦅
  #529 | 2026-03-06 | 4.1M | L7 | L/C 49:1 | Justice the 